![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/structural-break-real-time/assets/banner.webp)

# ADIA Lab Structural Break Challenge: Real-Time Edition

**A hands-on tutorial for participants.**

Welcome! In this notebook you will:

1. Learn **what the challenge is about**, in plain English.
2. See **what your code has to produce**, for every time series.
3. Understand **how submissions are scored** (the Time-Stratified AUC).
4. Walk through a **minimal working submission**: a simple streaming detector that you can submit as-is, then improve step by step.
5. Test your submission **locally**, so you can iterate quickly before uploading.

No prior experience with structural-break detection is required. If you can read Python and know what a time series is, you are in the right place.

## What is a structural break?

A **structural break** in a time series is a point in time at which the *rules* that generate the data change permanently. Before that moment the series follows one pattern; from that moment on, it follows a different one.

A simple example of a mean-shift break: the series oscillates around a lower mean before the break and around a higher one after.

![Example: a time series with a structural break. The dashed line marks the break.](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/structural-break-real-time/quickstarters/baseline/images/dgp_example.png)

Examples from the real world:

- **Finance.** A trading strategy that used to produce small, steady profits starts producing small, steady losses after some market regime change.
- **Industry.** A sensor on a machine used to oscillate around a fixed mean with fixed variance; after a bearing starts wearing out, the oscillations become larger.
- **Climate.** The average temperature in a region is stable for decades, then shifts upward as the local climate changes.
- **Healthcare.** A patient's resting heart-rate is steady for months, then drifts upward after a new medication is introduced.

The simplest kind of break is a *mean shift*: the average value of the series suddenly moves. <br />
But breaks can also be changes in variance (the series becomes noisier), in distribution (the shape of the noise changes), or in dependence structure (how each observation relates to the previous ones).

## Anatomy of a time series in this challenge

Each time series you receive is divided into two parts:

- A **historical segment** (1,000 to 5,000 observations), given in full at the start. This is the *reference* behaviour -- guaranteed to contain **no break**.
- An **online segment** (10 to 1,000 observations), revealed to your code **one observation at a time**. This is the part under scrutiny.

![Time series structure: historical segment (blue), online segment before the break (green), online segment after the break (red). The dashed vertical line marks the start of the online segment; the dotted line marks the break.](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/structural-break-real-time/quickstarters/baseline/images/competition_overview.png)

In the online segment, one of two things happens:

- Either a **break occurs** at some unknown time step $\tau$, and the behaviour after $\tau$ differs from the historical segment.
- or **No break occurs** at all -- the series just continues to behave like its historical reference.

Your job is to figure out, live, whether a break has already happened.

> **‼ Contrast with the 2025 edition**
> ---
> If you participated last year, the break (when present) was always at the known boundary between the two segments, and you returned a single score per series. This year the break location is unknown and can be anywhere in the online segment, and you return one score **per online time step**.

## What your code must output

At every online observation $t$, your code produces **one score $s_t \in [0, 1]$**. The score expresses your cumulative belief that a break has **already occurred somewhere up to and including step $t$**:

- $s_t = 0$: you are fully confident **no break** has happened yet.
- $s_t = 1$: you are fully confident a break **has already happened**.
- Anything in between is a soft confidence.

The **ideal score sequence** looks like a step function:

- Before the break, $s_t = 0$.
- From the break time $\tau$ onward, $s_t = 1$.

If there is no break at all, the ideal sequence is zero throughout.

Your real submission will not produce a perfect step. A good submission is one that, on average, increases sharply around $\tau$ and stays low otherwise.

The figure below contrasts a participant's score (solid) against the ideal step (dashed). The shaded area between them reflects how early and cleanly the break was picked up -- smaller is better.

![Evaluation example: participant scores (solid) vs ideal step function (dashed). A good submission keeps the shaded area small.](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/structural-break-real-time/quickstarters/baseline/images/evaluation_example.png)

## How submissions are scored: Time-Stratified AUC

All submissions are graded with a single metric, **Time-Stratified AUC** (TS-AUC). The intuition is:

1. **At each online time step $t$**, look across *all series alive at that step*. Some of those series have already had their break ($\text{label}_t = 1$); the others have not ($\text{label}_t = 0$).
2. **Compute an ordinary AUC** at that step: how well do your scores rank the series with a break above thhose without a break?
3. **Take a weighted average** of those per-step AUCs across all time steps.

The formula is:

$$
\text{TS-AUC} \;=\; \frac{\sum_t w(t)\,\text{AUC}(t)}{\sum_t w(t)},
\qquad w(t) = n_\text{pos}(t)\cdot n_\text{neg}(t)
$$

where $w(t)$ is the number of positive-negative pairs at step $t$ (steps with only one kind of series contribute nothing).

**Range and reference values:**

- TS-AUC = 1.0 means perfect detection at every step.
- TS-AUC = 0.5 means random-guessing (and is exactly what you get if your score depends only on $t$ -- for example, always returning $s_t = t / T$).
- Higher is better.

### A tiny worked-out example

Imagine just three series, each of online length 4. Below are the scores produced by a participant. A *check-mark* marks which online steps are post-break (label 1); the rest are pre-break (label 0).

| series           | $t=0$ | $t=1$  | $t=2$  | $t=3$  |
| ---------------- | ----- | ------ | ------ | ------ |
| A (break at t=2) | 0.10  | 0.15   | 0.60 ✓ | 0.80 ✓ |
| B (break at t=1) | 0.05  | 0.30 ✓ | 0.55 ✓ | 0.70 ✓ |
| C (no break)     | 0.02  | 0.10   | 0.20   | 0.40   |

At $t=0$: all three are pre-break -- no positive/negative pair, so this step contributes nothing. At $t=1$: B is positive, A and C are negative; the AUC at this step measures whether B's score (0.30) is higher than A's (0.15) and C"s (0.10) -- it is, so AUC(t=1) = 1.0.
And so on for $t=2, 3$. The final TS-AUC is the weighted average of these per-step AUCs.

Note: because the metric only compares scores **across series at the same time step**, a score that grows purely with $t$ (like $s_t = t/T$) gains nothing. This rewards *genuine* change-detection.

# Getting set up

The first steps to get started are:

1. Get the setup command from the competition page below.
2. Paste the token into the cell that follows and run it.

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)


In [ ]:
# Install the Crunch CLI
%pip install crunch-cli --upgrade --quiet --progress-bar off

# Setup your local environment
!crunch setup-notebook structural-break-real-time aaaabbbbccccddddeeeeffff

# Your submission

A submission to this competition consists of two Python functions:

- `train(datasets, model_directory_path)`: optional; fit any model you want on the training data and save the result to disk.
- `infer(datasets, model_directory_path)`: a **generator** that yields one score per online observation, in order.

We will first import dependencies and load the data, then walk through each function.

In [ ]:
import math
import os
from typing import Iterable, List, Optional, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

## Looking at the data

`crunch_tools.load_data()` returns two objects:

- `train_data`: the training set. <br />
  Each element is a tuple `(dataset_id, x_historical, x_online, tau_index)` where:
  - `dataset_id` is an integer identifier,
  - `x_historical` is a 1-D numpy array of the historical segment,
  - `x_online` is a 1-D numpy array of the full online segment,
  - `tau_index` is the index within `x_online` at which the break happens, or `None` if the series has no break.
- `test_data`: the test set. <br />
  with the same layout but *without* `tau_index`, and with `x_online` that can only be **iterated once** in the cloud environment.

<br />

> **📝 Note**
> ---
> Locally (inside this notebook) you can re-read the test series as many times as you like while developing.

In [ ]:
# Load the data.
train_data, test_data = crunch_tools.load_data()

### A single training series

Let us look at one training series with a break, to get a feel for the
data.


In [ ]:
# Pick a training series that has a break.
for dataset_id, x_hist, x_online, tau in train_data:
    if tau is not None and 50 <= tau <= len(x_online) - 50:
        break

x_hist   = np.asarray(x_hist)
x_online = np.asarray(x_online)

print(f"      series id: {dataset_id}")
print(f"historical size: {len(x_hist)}")
print(f"    online size: {len(x_online)}")
print(f"   break at tau: {tau}   (None means no break)")

      series id: 0
historical size: 1192
    online size: 416
   break at tau: 72   (None means no break)


## A baseline strategy: streaming EWMA z-score

Now that we have seen the data, we can design a simple detector. The baseline uses a classic idea from statistical process control:

1. Summarize the historical segment by its **mean** $\mu_H$ and **standard deviation** $\sigma_H$.
2. As online observations arrive, maintain an **exponentially-weighted moving average (EWMA)** $\bar{x}_t$ of the online values.
3. Compare $\bar{x}_t$ to the historical mean in *standard-deviation units*:

   $$
   z_t \;=\; \frac{\bar{x}_t - \mu_H}{\sigma_H / \sqrt{n_\text{eff}}}
   $$

   where $n_\text{eff}$ is the effective sample size of the EWMA.
4. Map $|z_t|$ to a score in $[0, 1]$ with a **tanh squash**:

   $$
   s_t \;=\; \tanh(|z_t|)
   $$

### Why this is a reasonable baseline

- **It is truly streaming.** <br />
  Each new point updates the EWMA with a constant amount of work; nothing is recomputed from scratch. <br />
  This is important -- with 10,000 series and up to 1,000 online steps each, a submission that re-fits at every step runs out of time.
- **It does not saturate.** <br />
  The score stays strictly below 1, so the detector can still distinguish *very strong* evidence from *strong* evidence.
- **It only reacts to mean shifts.** <br />
  Variance and shape changes are not captured well -- an opportunity for you to improve.

<br />

> **📝 Note (on what this baseline ignores)**
> ---
> It treats each series independently, which is exactly how it must -- the online segments arrive one at a time and there is no cross-series information available in production. <br />
> Incrementality is therefore not optional; it is built into the evaluation protocol.

### The `train()` function

`train()` is called once before any prediction is made. <br />
Its job is to fit whatever model you want and save it to `model_directory_path`.

The baseline does not need training -- the EWMA z-score is computed entirely from the historical segment and the streaming online values, so there is no model to fit. <br />
We simply save a placeholder so `infer()` has something to load.

If you later decide to use a supervized learner (e.g. a gradient boosting model that takes features of the running state and predicts the break probability), this is the function where you would fit and save it.

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    """
    Fit a model.
    """

    # The baseline has no parameters, so we just save an empty placeholder artefact.
    model = {
        "note": "EWMA z-score detector has no learned parameters"
    }

    for dataset_id, x_hist, x_online, tau in datasets:
        ...

    joblib.dump(model, os.path.join(model_directory_path, "model.joblib"))

### The `infer()` function

`infer()` is a **generator**. It must:

1. Load any model saved by `train()`.
2. `yield` once, with no value, to signal readiness to the runner.
3. For each test series `(x_historical, x_online)`:
    - Pre-compute whatever static summaries you need from `x_historical` (it is given in full and cheap to scan once).
    - Loop over the points of `x_online`. After each point, `yield` a `float` in $[0, 1]$ -- **exactly one yield per online point**, in order.

<br />

> **‼ Important**
> ---
> Both the outer `datasets` iterable and each `x_online` can be iterated **only once** in the cloud environment. <br />
> You must produce the score for the current observation before the next one is released.

### The streaming EWMA detector in code

We keep three running scalars per series:

- `mu_ewma`: the EWMA of the online values (tracks the current local mean).
- `n_eff`: the effective sample size of the EWMA (grows as more points arrive, bounded by `1 / (1 - alpha)`).
- `t`: the step index, for optional diagnostics.

At each new observation `x_t` we update these in O(1) and emit the score.

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    """
    Stream per-step break probabilities using an EWMA z-score.

    alpha -- EWMA decay in (0, 1]; higher = faster response but more noise.
    kappa -- tanh scale
    """

    ALPHA = 0.05    # effective window ~ 1/alpha = 20 points
    KAPPA = 3.0     # |z| = 3 -> score ~ 0.76

    # Load the model artefact.
    # We don"t actually have any parameters, but this shows how you would load them if you did.
    model = joblib.load(os.path.join(model_directory_path, "model.joblib"))

    yield  # Signal readiness to the runner.

    for x_historical, x_online in datasets:
        # Summarize the historical segment once.
        x_h = np.asarray(x_historical, dtype=np.float64)
        mu_h = float(x_h.mean()) if len(x_h) else 0.0
        sd_h = float(x_h.std(ddof=1)) if len(x_h) > 1 else 1.0
        sd_h = max(sd_h, 1e-8)  # avoid divide-by-zero

        # Streaming state, one scalar each.
        mu_ewma = mu_h    # Start centered on the historical mean.
        n_eff   = 0.0     # Effective sample size of the EWMA.

        for point in x_online:
            x = float(point)

            # O(1) EWMA update.
            mu_ewma = (1.0 - ALPHA) * mu_ewma + ALPHA * x
            n_eff   = (1.0 - ALPHA) * n_eff   + 1.0  # Grows to 1/alpha.

            # z-score of the running EWMA mean vs the historical mean.
            se = sd_h / math.sqrt(max(n_eff, 1.0))
            z  = (mu_ewma - mu_h) / max(se, 1e-8)

            # Squash to [0, 1] without saturating.
            score = math.tanh(abs(z) / KAPPA)

            yield float(score)

### Parallelism

If your model is capable of running in parallel, you should try enabling the parallelism mechanism. You can [learn more in the documentation](https://docs.crunchdao.com/competitions/competitions/structural-break-real-time#parallelism).

But the TL;DR is:
- Your model will be run N times, with each call happening in a different process (rather than a thread).

- To avoid concurrency issues, do not write file from the `infer()` function.

- Make sure you don't use too many resources. <br />
  - If your model requires 4 GB of RAM and you want a parallelism of 6, the machine will need at least 24 GB of RAM (+overhead).
  - The same applies to CPUs: try not to use more than the number of CPUs your machine has (+overhead).

- It makes debugging more difficult: if you need to diagnose a bug, revert to a value of `1`.

In [ ]:
# @crunch/keep:on
INFER_PARALLELISM = 4

# Uncomment this line to run `infer` with maximum parallelism while leaving one CPU core free for other tasks
# INFER_PARALLELISM = os.cpu_count() - 1

# Uncomment this line to run infer without parallelism
# INFER_PARALLELISM = 1

## Local testing

The Crunch CLI ships with a local tester that reproduces the cloud
environment. <br />
It calls `train()` once, then calls `infer()` with the test data, collects the yielded scores, and writes them to `prediction/prediction.parquet`.

This is the same flow the platform will run.

In [ ]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

## Previewing the results

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction.head(10)

prediction
id    time            
10000 3312    0.002722
      3313    0.001283
      3314    0.024189
      3315    0.045730
      3316    0.075652
      3317    0.092703
      3318    0.102657
      3319    0.152654
      3320    0.123343
      3321    0.045093

### Computing TS-AUC locally

Below is a straightforward implementation of the TS-AUC metric: group rows by online time step, compute the AUC cross-sectionally at each step (skipping steps that only have one class), and return the weighted average.

This is exactly what the leaderboard uses, just without the private test set.

In [ ]:
# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True,
)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

# Weighted per-step AUC.
weighted_auc_sum = 0.0
total_weight     = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t  = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight     += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

Local TS-AUC: 0.5170


## Ideas for stronger solutions

The baseline above is a starting point, not a competitive submission. <br />
Here are directions that typically pay off, in rough order of effort vs. expected benefit:

**Richer streaming statistics.** The EWMA reacts to *mean* shifts but ignores changes in variance, distributional shape, and serial correlation. Try maintaining, in parallel:

- A **CUSUM** of standardized residuals (cumulative deviation from the historical mean), which accumulates evidence over time rather than decaying it.
- Running **variance** and a ratio against $\sigma_H^2$ to catch volatility shifts.
- Fraction of online points beyond $\pm 2\sigma_H$ or $\pm 3\sigma_H$ (tail-mass change).
- Online **autocorrelation** at a few lags, compared to the historical autocorrelation.
- A **likelihood-ratio CUSUM** under a small autoregressive model fit to the historical segment.

All of these can be updated in O(1) per new observation with a bit of care.

**Combine features with a supervized model:**
- If you collect a handful of incremental features like the ones above into a vector, you have a standard tabular classification problem at every online step.
- A gradient boosting model (LightGBM, XGBoost) trained on `(feature_vector, label_t)` pairs from the training set can learn to weight the features much better than a hand-tuned rule.
- Remember to store *all* the state needed to reproduce the feature vector at inference time in O(1).

**Watch your time budget:**
- With 10,000 series and up to 1,000 online steps each, your code may be asked for up to 10 million scores.
- At any step, anything slower than a few microseconds of Python is a liability. Profile!

# Submitting your notebook

To submit your work, you must:

1. Download your notebook (from Colab, Kaggle, or a local copy).
2. Upload it to the competition platform.
3. Create a **run** to validate it against the full test set.

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)